# Vision Transformers (ViT)

## Learning Objectives
1. Implement patch extraction and positional encoding from scratch
2. Build a minimal ViT classifier in PyTorch with multi-head self-attention
3. Visualize attention maps to understand what patches the model focuses on
4. Compare CNN vs ViT data efficiency on a small synthetic dataset


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Level 1: Patch Extraction and Positional Encoding from Scratch

In [ ]:
# ---- Level 1: Patch extraction and positional encodings in numpy ----------
# ViT splits the image into a grid of non-overlapping patches,
# flattens each patch, and projects it to a D-dimensional embedding.
# Positional encoding tells the model WHERE each patch came from.


def extract_patches(image, patch_size):
    """Split a 2D image into non-overlapping patches.

    Args:
        image: numpy array (H, W) or (C, H, W)
        patch_size: int P — patches are P x P pixels

    Returns:
        patches: (N, patch_size*patch_size*C) where N = (H*W) / (P*P)
        positions: (N, 2) row/column grid position of each patch
    """
    if image.ndim == 2:
        image = image[np.newaxis]  # add channel dim
    C, H, W = image.shape
    assert H % patch_size == 0, 'H must be divisible by patch_size'
    assert W % patch_size == 0, 'W must be divisible by patch_size'
    n_row = H // patch_size
    n_col = W // patch_size
    patches = []
    positions = []
    for row in range(n_row):
        for col in range(n_col):
            # Extract PxP patch from each channel and flatten
            patch = image[
                :,
                row * patch_size:(row + 1) * patch_size,
                col * patch_size:(col + 1) * patch_size,
            ]
            patches.append(patch.flatten())  # C*P*P
            positions.append([row, col])
    return np.array(patches), np.array(positions)


def sinusoidal_positional_encoding(n_positions, d_model):
    """1D sinusoidal positional encoding (from 'Attention is All You Need').

    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

    This encoding allows the model to generalize to sequence lengths not
    seen during training by using smooth interpolation.
    """
    PE = np.zeros((n_positions, d_model))
    positions = np.arange(n_positions).reshape(-1, 1)  # (N, 1)
    # Frequency denominators: 10000^(2i/d_model)
    div_term = np.power(
        10000.0, np.arange(0, d_model, 2) / d_model
    )
    PE[:, 0::2] = np.sin(positions / div_term)  # even dims: sine
    PE[:, 1::2] = np.cos(positions / div_term[:d_model // 2])  # odd dims: cosine
    return PE


def learned_positional_encoding_demo(n_patches, d_model):
    """Learned PE: simple random initialization (trained via gradient descent).

    In practice: nn.Parameter(torch.randn(1, n_patches+1, d_model)) where
    +1 accounts for the CLS token. The model adjusts these during training.
    """
    return np.random.randn(n_patches, d_model) * 0.02  # small init


# Demonstrate on a 32x32 image with 8x8 patches -> 16 patches
sample_image = np.random.randn(3, 32, 32).astype(np.float32)
patches, positions = extract_patches(sample_image, patch_size=8)

print(f'Image shape: {sample_image.shape}')
print(f'Patch size: 8x8 pixels -> {patches.shape[1]} values per patch')
print(f'Number of patches: {patches.shape[0]} (= 4x4 grid of 8x8 patches)')
print(f'Position array: {positions.shape} (row, col for each patch)')

# Show PE pattern: each position gets a unique encoding
D_MODEL = 64
sin_pe = sinusoidal_positional_encoding(16, D_MODEL)
learned_pe = learned_positional_encoding_demo(16, D_MODEL)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.imshow(sin_pe, aspect='auto', cmap='RdBu_r')
ax1.set_title('Sinusoidal PE (16 patches x 64 dims)')
ax1.set_xlabel('Embedding dimension')
ax1.set_ylabel('Patch position')
ax2.imshow(learned_pe, aspect='auto', cmap='RdBu_r')
ax2.set_title('Learned PE initialization (before training)')
ax2.set_xlabel('Embedding dimension')
ax2.set_ylabel('Patch position')
plt.suptitle('Level 1: Positional Encoding for 16 Patches (8x8 on 32x32 image)')
plt.tight_layout()
plt.savefig('/tmp/cv04_pe.png', dpi=80)
plt.close()
print('Saved: /tmp/cv04_pe.png')
print('Sinusoidal PE: smooth patterns. Nearby positions are more similar.')


## Level 2: Minimal ViT in PyTorch

In [ ]:
# ---- Level 2: Minimal ViT for image classification ----------------------
# Full ViT pipeline: patch embed -> CLS token -> positional encoding
# -> L transformer blocks -> CLS output -> classifier


class PatchEmbedding(nn.Module):
    """Linear projection from flattened patches to D-dimensional embeddings.

    Equivalent to applying a strided convolution with kernel_size=patch_size.
    Both forms produce the same result; linear projection is more explicit.
    """

    def __init__(self, img_size, patch_size, in_ch, embed_dim):
        super().__init__()
        assert img_size % patch_size == 0
        self.n_patches = (img_size // patch_size) ** 2
        self.patch_dim = in_ch * patch_size * patch_size
        # One linear layer projects each flattened patch to embed_dim
        self.projection = nn.Linear(self.patch_dim, embed_dim)
        self.patch_size = patch_size
        self.img_size = img_size
        self.in_ch = in_ch

    def forward(self, x):
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        P = self.patch_size
        # Reshape into patches: (B, N, patch_dim)
        x = x.reshape(B, C, H // P, P, W // P, P)
        x = x.permute(0, 2, 4, 1, 3, 5)  # (B, nH, nW, C, P, P)
        x = x.reshape(B, self.n_patches, -1)  # (B, N, C*P*P)
        return self.projection(x)             # (B, N, embed_dim)


class TransformerBlock(nn.Module):
    """One transformer encoder block: Pre-LN MHA + FFN.

    Pre-LN (LayerNorm before attention) is more stable than post-LN
    and does not require careful learning rate warmup.
    """

    def __init__(self, embed_dim, n_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        # Multi-head self-attention: each head sees embed_dim/n_heads features
        self.attn = nn.MultiheadAttention(
            embed_dim, n_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        # FFN expands to 4x then contracts back — standard transformer pattern
        mlp_dim = int(embed_dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Pre-LN attention with residual
        normed = self.norm1(x)
        attn_out, attn_weights = self.attn(
            normed, normed, normed, need_weights=True, average_attn_weights=False
        )
        x = x + attn_out             # residual connection
        x = x + self.ffn(self.norm2(x))  # FFN with residual
        return x, attn_weights          # return weights for visualization


class MiniViT(nn.Module):
    """Minimal Vision Transformer for small image classification.

    Architecture:
    - Patch embedding: img -> (N, embed_dim) sequence
    - CLS token: prepended learnable vector for classification
    - Learned positional encoding: added to all tokens
    - L transformer blocks
    - CLS token output -> classifier head
    """

    def __init__(self, img_size=32, patch_size=8, in_ch=3, embed_dim=128,
                 n_layers=4, n_heads=4, num_classes=10, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_ch, embed_dim)
        n_patches = self.patch_embed.n_patches
        # CLS token: one extra learnable token at position 0
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        # Positional encoding: n_patches + 1 (for CLS) learnable vectors
        self.pos_encoding = nn.Parameter(
            torch.randn(1, n_patches + 1, embed_dim) * 0.02
        )
        self.dropout = nn.Dropout(dropout)
        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, n_heads, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_encoding, std=0.02)

    def forward(self, x, return_attn=False):
        B = x.shape[0]
        # Patch embedding: (B, N, embed_dim)
        tokens = self.patch_embed(x)
        # Expand CLS token to batch size and prepend
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        tokens = torch.cat([cls, tokens], dim=1)  # (B, N+1, embed_dim)
        # Add positional encoding: critical for spatial awareness
        tokens = self.dropout(tokens + self.pos_encoding)
        # Transformer blocks
        attn_maps = []
        for block in self.blocks:
            tokens, attn_w = block(tokens)
            attn_maps.append(attn_w)
        # Classification: extract CLS token output (position 0)
        cls_out = self.norm(tokens[:, 0])  # (B, embed_dim)
        logits = self.classifier(cls_out)
        if return_attn:
            return logits, attn_maps
        return logits


# Generate synthetic CIFAR-like data for ViT
N_SAMPLES = 1200
N_CLASSES = 10
IMG_SIZE = 32

X_vit = torch.zeros(N_SAMPLES, 3, IMG_SIZE, IMG_SIZE)
y_vit = torch.zeros(N_SAMPLES, dtype=torch.long)
for cls in range(N_CLASSES):
    n = N_SAMPLES // N_CLASSES
    start = cls * n
    # Class-specific spatial patterns to give ViT's attention something to find
    base = torch.randn(n, 3, IMG_SIZE, IMG_SIZE) * 0.3
    color = torch.tensor([cls / N_CLASSES, cls / (2*N_CLASSES), 0.5]).view(1,3,1,1)
    X_vit[start:start+n] = base + color
    y_vit[start:start+n] = cls

mean_v = X_vit.mean(dim=(0,2,3), keepdim=True)
std_v = X_vit.std(dim=(0,2,3), keepdim=True) + 1e-7
X_vit = (X_vit - mean_v) / std_v

vit_ds = TensorDataset(X_vit, y_vit)
vit_train, vit_val = random_split(vit_ds, [1000, 200])
vit_loader = DataLoader(vit_train, batch_size=64, shuffle=True)
vit_val_loader = DataLoader(vit_val, batch_size=64)

# Train minimal ViT
vit_model = MiniViT(
    img_size=IMG_SIZE, patch_size=8, in_ch=3, embed_dim=128,
    n_layers=4, n_heads=4, num_classes=N_CLASSES
).to(device)
vit_opt = optim.AdamW(vit_model.parameters(), lr=3e-4, weight_decay=0.05)
vit_sched = optim.lr_scheduler.CosineAnnealingLR(vit_opt, T_max=30)
crit = nn.CrossEntropyLoss()

vit_train_losses, vit_val_accs = [], []
VIT_EPOCHS = 30
for epoch in range(VIT_EPOCHS):
    vit_model.train()
    ep_loss = 0.0
    for X_b, y_b in vit_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        vit_opt.zero_grad()
        try:
            logits = vit_model(X_b)
            loss = crit(logits, y_b)
            loss.backward()
            # Gradient clipping is important for ViT stability early in training
            torch.nn.utils.clip_grad_norm_(vit_model.parameters(), max_norm=1.0)
            vit_opt.step()
            ep_loss += loss.item()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                print('OOM: reduce batch_size or embed_dim')
            else:
                raise
    vit_sched.step()
    vit_train_losses.append(ep_loss / len(vit_loader))
    if (epoch + 1) % 10 == 0:
        vit_model.eval()
        correct = 0
        with torch.no_grad():
            for X_v, y_v in vit_val_loader:
                correct += (vit_model(X_v.to(device)).argmax(1)
                            == y_v.to(device)).sum().item()
        val_acc = correct / 200
        vit_val_accs.append((epoch + 1, val_acc))
        print(f'Epoch {epoch+1}: val_acc = {val_acc:.3f}')

vit_params = sum(p.numel() for p in vit_model.parameters())
print(f'ViT params: {vit_params:,}')


## Real-World Example 1: Attention Map Visualization

In [ ]:
# ---- RW1: Attention map visualization — what patches does ViT attend to? --
# The CLS token's attention weights across patch tokens show which image
# regions the model uses for its classification decision.


def get_cls_attention(model, img_tensor):
    """Extract CLS-to-patch attention weights from last transformer block.

    Args:
        model: MiniViT in eval mode
        img_tensor: (1, C, H, W) single image batch

    Returns:
        attn_map: (n_heads, n_patches) attention from CLS to all patches
    """
    model.eval()
    with torch.no_grad():
        _, attn_maps = model(img_tensor.to(device), return_attn=True)
    # Last block attention: shape (B, n_heads, seq_len, seq_len)
    last_attn = attn_maps[-1][0]  # (n_heads, seq_len, seq_len)
    # CLS token is at position 0 — its attention to patches at positions 1:
    cls_attn = last_attn[:, 0, 1:]  # (n_heads, n_patches)
    return cls_attn.cpu().numpy()


# Visualize attention maps for a few samples
n_patches_per_side = IMG_SIZE // 8  # 32/8 = 4
n_patches = n_patches_per_side ** 2  # 16
n_heads = 4

fig, axes = plt.subplots(n_heads + 1, 4, figsize=(14, 14))

for sample_idx in range(4):
    img_t = X_vit[sample_idx:sample_idx + 1]
    cls_attn = get_cls_attention(vit_model, img_t)
    # Display image (un-normalize for display)
    img_display = img_t[0].permute(1, 2, 0).numpy()
    img_display = (img_display - img_display.min()) / (
        img_display.max() - img_display.min() + 1e-7
    )
    axes[0, sample_idx].imshow(img_display)
    axes[0, sample_idx].set_title(
        f'Sample {sample_idx+1}\nClass {y_vit[sample_idx].item()}'
    )
    axes[0, sample_idx].axis('off')
    # Plot each head's attention map
    for head_idx in range(n_heads):
        # Reshape flat n_patches -> (n_patches_per_side, n_patches_per_side)
        attn_grid = cls_attn[head_idx].reshape(
            n_patches_per_side, n_patches_per_side
        )
        axes[head_idx + 1, sample_idx].imshow(attn_grid, cmap='hot')
        axes[head_idx + 1, sample_idx].set_title(f'Head {head_idx+1}')
        axes[head_idx + 1, sample_idx].axis('off')

axes[0, 0].set_ylabel('Input', rotation=90, labelpad=5)
for h in range(n_heads):
    axes[h + 1, 0].set_ylabel(f'Head {h+1}', rotation=90, labelpad=5)

plt.suptitle('RW1: CLS Token Attention per Head (hot=high attention)')
plt.tight_layout()
plt.savefig('/tmp/cv04_attention.png', dpi=80)
plt.close()
print('Saved: /tmp/cv04_attention.png')
print('Each head specializes in attending to different patch regions.')
print('High attention = the CLS token relies heavily on those patches.')


## Real-World Example 2: Positional Encoding Ablation

In [ ]:
# ---- RW2: Positional encoding ablation — what happens without it? --------
# Without PE, the model is permutation-invariant: it cannot distinguish
# which patch is top-left vs bottom-right. This hurts spatially-dependent tasks.


class MiniViTNoPE(MiniViT):
    """ViT variant with positional encoding zeroed out during training.

    This ablation shows how much spatial information PE provides.
    Without PE, the model sees a bag of patches with no positional ordering.
    """

    def forward(self, x, return_attn=False):
        B = x.shape[0]
        tokens = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        # Key difference: do NOT add positional encoding
        tokens = self.dropout(tokens)  # no pos_encoding added
        attn_maps = []
        for block in self.blocks:
            tokens, attn_w = block(tokens)
            attn_maps.append(attn_w)
        cls_out = self.norm(tokens[:, 0])
        logits = self.classifier(cls_out)
        if return_attn:
            return logits, attn_maps
        return logits


def train_vit(ModelClass, tag, epochs=25):
    """Train a ViT variant and return validation accuracy per epoch."""
    m = ModelClass(
        img_size=IMG_SIZE, patch_size=8, in_ch=3, embed_dim=128,
        n_layers=4, n_heads=4, num_classes=N_CLASSES
    ).to(device)
    opt = optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.05)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    val_accs = []
    for epoch in range(epochs):
        m.train()
        for X_b, y_b in vit_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            opt.zero_grad()
            loss = criterion(m(X_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
        sched.step()
        m.eval()
        correct = 0
        with torch.no_grad():
            for X_v, y_v in vit_val_loader:
                correct += (m(X_v.to(device)).argmax(1)
                            == y_v.to(device)).sum().item()
        val_accs.append(correct / 200)
    print(f'{tag}: final val_acc = {val_accs[-1]:.3f}')
    return val_accs


vit_with_pe = train_vit(MiniViT, tag='ViT with PE')
vit_no_pe = train_vit(MiniViTNoPE, tag='ViT without PE')

plt.figure(figsize=(9, 4))
plt.plot(vit_with_pe, label='ViT with positional encoding', color='steelblue')
plt.plot(vit_no_pe, label='ViT without positional encoding', color='tomato',
         linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('Positional Encoding Ablation: PE is Critical for Spatial Tasks')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/cv04_pe_ablation.png', dpi=80)
plt.close()
print('Saved: /tmp/cv04_pe_ablation.png')


## Real-World Example 3: Patch Size Effect on Sequence Length

## Comparison: CNN vs ViT Data Efficiency

In [ ]:
# ---- RW3 + Comparison: Patch size effect + CNN vs ViT efficiency ----------
# Smaller patches -> longer sequence -> more context but O(N^2) cost.
# ViT needs more data than CNN because it has no locality inductive bias.


def build_vit_with_patch(patch_size):
    """Build a ViT with the given patch size on 32x32 images."""
    return MiniViT(
        img_size=IMG_SIZE, patch_size=patch_size, in_ch=3,
        embed_dim=128, n_layers=3, n_heads=4, num_classes=N_CLASSES
    ).to(device)


# Patch size impact on sequence length and parameter count
print('Patch size vs sequence length vs parameters:')
patch_configs = []
for ps in [4, 8, 16]:
    m = build_vit_with_patch(ps)
    n_seq = (IMG_SIZE // ps) ** 2
    n_params = sum(p.numel() for p in m.parameters())
    print(f'  Patch {ps}x{ps}: sequence_len={n_seq:3d}, '
          f'attn_cost=O({n_seq}^2)={n_seq**2:5d}, params={n_params:,}')
    patch_configs.append((ps, n_seq, n_seq**2, n_params))

# CNN vs ViT on varying training set sizes — data efficiency comparison
# Reuse SmallResNet from notebook 01 as CNN baseline
class SmallResNetForViT(nn.Module):
    """Compact ResNet-style CNN with same parameter budget as the mini ViT."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(128, num_classes)

    def forward(self, x):
        return self.head(self.gap(self.features(x)).flatten(1))


def train_data_efficiency(ModelClass, data_fractions, tag, epochs=20):
    """Train with different fractions of training data; return final val accuracies.

    Tests the hypothesis: CNNs learn from less data than ViTs because
    convolution provides strong spatial inductive bias.
    """
    accs = []
    full_X, full_y = X_vit[:1000], y_vit[:1000]
    for frac in data_fractions:
        n_use = max(64, int(1000 * frac))  # at least 64 samples
        sub_X, sub_y = full_X[:n_use], full_y[:n_use]
        loader = DataLoader(TensorDataset(sub_X, sub_y),
                           batch_size=min(64, n_use), shuffle=True)
        m = ModelClass(num_classes=N_CLASSES).to(device)
        opt = optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.05)
        crit_fn = nn.CrossEntropyLoss()
        for epoch in range(epochs):
            m.train()
            for X_b, y_b in loader:
                opt.zero_grad()
                loss = crit_fn(m(X_b.to(device)), y_b.to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
        m.eval()
        correct = 0
        with torch.no_grad():
            for X_v, y_v in vit_val_loader:
                correct += (m(X_v.to(device)).argmax(1)
                            == y_v.to(device)).sum().item()
        accs.append(correct / 200)
    print(f'{tag} accs: {[round(a,3) for a in accs]}')
    return accs


fractions = [0.1, 0.2, 0.4, 0.7, 1.0]
cnn_accs = train_data_efficiency(SmallResNetForViT, fractions, 'CNN')
vit_accs = train_data_efficiency(MiniViT, fractions, 'ViT')

n_samples_list = [int(f * 1000) for f in fractions]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CNN vs ViT data efficiency
axes[0].plot(n_samples_list, cnn_accs, 'o-', label='CNN (inductive bias)',
             color='steelblue')
axes[0].plot(n_samples_list, vit_accs, 's--', label='ViT (no inductive bias)',
             color='tomato')
axes[0].set_xlabel('Training set size')
axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('CNN vs ViT: Data Efficiency')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Patch size vs attention cost
ps_vals = [c[0] for c in patch_configs]
attn_costs = [c[2] for c in patch_configs]
axes[1].bar([f'{p}x{p}' for p in ps_vals], attn_costs,
            color=['steelblue', 'orange', 'tomato'])
axes[1].set_xlabel('Patch Size')
axes[1].set_ylabel('Attention cost O(N^2)')
axes[1].set_title('Patch Size vs Attention Cost (quadratic in N)')
for i, (v, cost) in enumerate(zip(ps_vals, attn_costs)):
    axes[1].text(i, cost + 5, str(cost), ha='center', fontsize=10)
axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('Comparison: ViT Data Efficiency and Patch Size Trade-offs')
plt.tight_layout()
plt.savefig('/tmp/cv04_comparison.png', dpi=80)
plt.close()
print('Saved: /tmp/cv04_comparison.png')
print('CNN typically outperforms ViT at small data sizes.')
print('ViT catches up with more data because global attention is more powerful.')
